In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

df_bronze = spark.read.format("delta").table("bronze_transactions")

df_silver = df_bronze.filter(F.col("amount") > 0)
df_silver = df_silver.dropDuplicates(["transaction_id"])
df_silver = df_silver.withColumn("amount", F.col("amount").cast(DoubleType()))
df_silver = df_silver.withColumn("city", F.initcap(F.trim(F.col("city"))))

df_silver = df_silver.withColumn(
    "is_weekend",
    F.when(F.col("day_of_week").isin("Saturday", "Sunday"), 1).otherwise(0)
)
df_silver = df_silver.withColumn(
    "amount_category",
    F.when(F.col("amount") < 500,   "Low")
     .when(F.col("amount") < 5000,  "Medium")
     .when(F.col("amount") < 20000, "High")
     .otherwise("Very High")
)
df_silver = df_silver.withColumn(
    "is_late_night",
    F.when(
        (F.col("hour_of_day") >= 23) | (F.col("hour_of_day") <= 5), 1
    ).otherwise(0)
)
df_silver = df_silver.withColumn("processed_at", F.current_timestamp())

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_transactions")

print("Silver layer complete!")